# 🐍 VenomSearch-AI — Fase 1: Comprensión de los Datos y EDA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Franco-Arce/venomsearch-ai/blob/main/notebooks/01_eda_and_umap.ipynb)

**Proyecto:** VenomSearch-AI — Búsqueda Vectorial Inteligente de Neurotoxinas  
**Dataset:** UniProt Tox-Prot (Proteínas tóxicas revisadas de Swiss-Prot)  
**Objetivo de la Fase 1:** Análisis Exploratorio de Datos (EDA), caracterización del dataset, clasificación de variables numéricas y categóricas, análisis estadístico descriptivo y visualizaciones de hallazgos preliminares.

---

## Estructura del Informe de Fase 1
1. **Configuración del Entorno (Local / Google Colab)**
2. **Comprensión del Negocio y Origen de Datos**
3. **Carga y Calidad del Dataset** (Registros, Nulos, Duplicados)
4. **Identificación y Clasificación Inicial de Variables** (Numéricas vs. Categóricas)
5. **Análisis Estadístico Descriptivo**
6. **Análisis Exploratorio Gráfico (Visualizaciones)**
7. **Proyección en Espacio Latente (PCA + UMAP)**
8. **Hallazgos Preliminares y Conclusiones**

## 1. Configuración de Entorno e Importación de Librerías

Detección automática de Google Colab: instala automáticamente las librerías necesarias y descarga los datos procesados si se ejecuta en la nube.

In [ ]:
import sys
import os

# Detección y auto-setup para Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('🚀 Ejecutando en Google Colab — Configurando dependencias y repositorio...')
    !pip install -q polars umap-learn plotly scikit-learn seaborn matplotlib
    if not os.path.exists('venomsearch-ai'):
        !git clone https://github.com/Franco-Arce/venomsearch-ai.git
    os.chdir('venomsearch-ai/notebooks')

import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.decomposition import PCA
import umap
from sklearn.metrics.pairwise import cosine_similarity

# Configuración de estilo gráfico
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
sns.set_palette('muted')
%matplotlib inline
print('✅ Entorno e importaciones listos.')

## 2. Carga del Dataset y Evaluación de Calidad de Datos

Cargamos la tabla estructurada resultante de la ingesta desde la API REST de UniProt (`toxins.parquet`) y la matriz de embeddings (`embeddings.npy`) generada por el modelo de lenguaje de proteínas ESM-2.

In [ ]:
# Verificación dinámica de rutas (soporta local y Colab)
data_dir = '../data/processed'
if not os.path.exists(data_dir) and os.path.exists('data/processed'):
    data_dir = 'data/processed'

df = pl.read_parquet(os.path.join(data_dir, 'toxins.parquet'))
embeddings = np.load(os.path.join(data_dir, 'embeddings.npy'))

print('==================================================')
print(f'Dimensiones del Dataset Tabular: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'Dimensiones de la Matriz de Embeddings: {embeddings.shape[0]} secuencias x {embeddings.shape[1]} dimensiones')
print('==================================================')

In [ ]:
# Verificación de integridad: Nulos y Duplicados
null_counts = df.null_count()
duplicates = df.shape[0] - df['accession'].n_unique()

print('--- CONTEO DE VALORES NULOS POR COLUMNA ---')
print(null_counts.to_pandas().T.rename(columns={0: 'nulos'}))
print(f'\nTotal de registros duplicados por Accession ID: {duplicates}')

## 3. Identificación y Clasificación Inicial de Variables

De acuerdo a los requerimientos de la **Fase 1**, a continuación se detalla la taxonomía y tipo de dato de cada una de las variables presentes en el dataset:

| Variable | Tipo de Dato (Python/Polars) | Clasificación Estadística | Descripción / Rol | Ejemplo |
| :--- | :--- | :--- | :--- | :--- |
| `accession` | `String` | **Cualitativa Nominal** | Identificador único de proteína en UniProt (Clave primaria) | `P01375` |
| `entry_name` | `String` | **Cualitativa Nominal** | Nombre nemotécnico de entrada | `CX3A_CONGE` |
| `protein_name` | `String` | **Cualitativa Nominal** | Nombre completo/descriptivo de la toxina | `Alpha-conotoxin GI` |
| `organism` | `String` | **Cualitativa Nominal** | Especie u organismo biológico de origen | `Conus geographus` |
| `organism_id` | `Int64` | **Cualitativa Nominal (ID)** | Identificador taxonómico de la especie (NCBI Taxonomy ID) | `6492` |
| `sequence` | `String` | **Cualitativa Nominal (Texto)** | Secuencia de aminoácidos primaria en código IUPAC de 1 letra | `ECCNPACGRHYSC*` |
| `sequence_length` | `Int64` | **Cuantitativa Discreta** | Cantidad total de aminoácidos que componen la proteína | `13` |
| `disulfide_bonds` | `Int64` | **Cuantitativa Discreta** | Cantidad de enlaces disulfuro (Cys-Cys) confirmados | `2` |
| `toxin_family` | `String` | **Cualitativa Nominal** | Familia funcional/estructural de la toxina (Target principal) | `Conotoxin` |
| `subcellular_location` | `String` | **Cualitativa Nominal** | Localización celular conocida | `Secreted` |
| `function_annotation` | `String` | **Cualitativa Nominal (Texto libre)** | Descripción funcional / mecanismo de toxicidad | `Neurotoxin that blocks...` |
| `is_reviewed` | `Boolean` | **Cualitativa Binaria** | Indicador de curación manual en Swiss-Prot (100% True) | `True` |
| `keywords` | `List(String)` | **Multivalor Categorica** | Palabras clave ontológicas de UniProt | `['Neurotoxin', 'Ion channel']` |
| `go_terms` | `List(String)` | **Multivalor Categorica** | Términos de Gene Ontology | `['GO:0006811']` |
| `embeddings` (Numpy) | `Float32 (156x320)` | **Cuantitativas Continuas** | Vector de 320 dimensiones latentes continuas generadas por ESM-2 | `[-0.042, 0.128, ...]` |

## 4. Análisis Estadístico Descriptivo

Presentación de métricas de tendencia central y dispersión para las variables numéricas clave.

In [ ]:
# Estadísticas descriptivas de variables cuantitativas
num_cols = ['sequence_length', 'disulfide_bonds']
summary_df = df.select(num_cols).to_pandas().describe().T

# Formatear resumen estadístico
summary_df['IQR'] = summary_df['75%'] - summary_df['25%']
print('--- RESUMEN ESTADÍSTICO DE VARIABLES NUMÉRICAS ---')
print(summary_df[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max', 'IQR']].round(2))

In [ ]:
# Frecuencias de variables categóricas principales
print('--- DISTRIBUCIÓN DE FAMILIAS DE TOXINAS (TOP) ---')
family_counts = df.group_by('toxin_family').len().sort('len', descending=True)
print(family_counts.to_pandas().head(10))

print('\n--- DIVERSIDAD ORGANÍSMICA ---')
print(f'Total de especies distintas: {df["organism"].n_unique()}')

## 5. Análisis Exploratorio Gráfico (Visualizaciones)

Visualización de distribuciones y patrones de las variables numéricas y categóricas.

In [ ]:
# Fig 1: Histograma + Boxplot de Longitud de Secuencia
fig, (ax_box, ax_hist) = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={'height_ratios': [0.2, 0.8]})

lengths = df['sequence_length'].to_numpy()
median_val = np.median(lengths)
mean_val = np.mean(lengths)

# Boxplot en la parte superior
sns.boxplot(x=lengths, ax=ax_box, color='#64B5F6', fliersize=4)
ax_box.set(xlabel='')
ax_box.set_title('Análisis de Distribución de Longitud de Secuencia (Aminoácidos)', fontsize=14, fontweight='bold')

# Histograma con KDE en la parte inferior
sns.histplot(lengths, kde=True, ax=ax_hist, color='#1E88E5', bins=40, alpha=0.7)
ax_hist.axvline(median_val, color='red', linestyle='--', linewidth=2, label=f'Mediana: {median_val:.0f} aa')
ax_hist.axvline(mean_val, color='orange', linestyle='-', linewidth=2, label=f'Media: {mean_val:.1f} aa')
ax_hist.set_xlabel('Longitud de Secuencia (Número de Aminoácidos)', fontsize=12)
ax_hist.set_ylabel('Frecuencia (N° de Proteínas)', fontsize=12)
ax_hist.legend(fontsize=11)

plt.tight_layout()
os.makedirs(data_dir, exist_ok=True)
plt.savefig(os.path.join(data_dir, 'length_distribution_eda.png'), dpi=150)
plt.show()

In [ ]:
# Fig 2: Top 15 Organismos Productores de Toxinas
top_orgs = df.group_by('organism').len().sort('len', descending=True).head(15).to_pandas()

plt.figure(figsize=(12, 6))
barplot = sns.barplot(
    data=top_orgs,
    y='organism',
    x='len',
    palette='viridis'
)
plt.title('Top 15 Organismos Productores de Toxinas en el Dataset', fontsize=14, fontweight='bold')
plt.xlabel('Número de Toxinas Registradas', fontsize=12)
plt.ylabel('Especie', fontsize=12)

# Añadir etiquetas numéricas sobre las barras
for p in barplot.patches:
    width = p.get_width()
    barplot.text(width + 0.3, p.get_y() + p.get_height()/2, f'{int(width)}', ha='left', va='center', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(data_dir, 'top_organisms_eda.png'), dpi=150)
plt.show()

In [ ]:
# Fig 3: Distribución de Familias de Toxinas
fam_df = df.group_by('toxin_family').len().sort('len', descending=True).to_pandas()

plt.figure(figsize=(10, 5))
fam_bar = sns.barplot(data=fam_df, x='len', y='toxin_family', palette='magma')
plt.title('Distribución de Proteínas por Familia de Toxina', fontsize=14, fontweight='bold')
plt.xlabel('Cantidad de Secuencias', fontsize=12)
plt.ylabel('Familia Funcional', fontsize=12)

for p in fam_bar.patches:
    width = p.get_width()
    fam_bar.text(width + 0.3, p.get_y() + p.get_height()/2, f'{int(width)}', ha='left', va='center', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(data_dir, 'toxin_families_eda.png'), dpi=150)
plt.show()

## 6. Proyección Espacial y Aglomeración No Supervisada (PCA + UMAP)

Para comprender la estructura subyacente del espacio continuo de 320 dimensiones (embeddings de ESM-2), reducimos dimensionalidad con PCA a 50 componentes y luego proyectamos a 2D utilizando UMAP con métrica de coseno.

In [ ]:
# Paso 1: Reducción de ruido previa con PCA (320 -> 50 dims)
pca = PCA(n_components=min(50, len(df)), random_state=42)
embeddings_pca = pca.fit_transform(embeddings)
var_explicada = pca.explained_variance_ratio_.sum()
print(f'Varianza explicada acumulada por 50 componentes principales: {var_explicada:.2%}')

# Paso 2: Proyección 2D con UMAP (Métrica Coseno)
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    metric='cosine',
    random_state=42
)
embeddings_2d = reducer.fit_transform(embeddings_pca)

In [ ]:
# Proyección 2D UMAP coloreada por Familia Funcional
plt.figure(figsize=(12, 8))

families = df['toxin_family'].to_list()
unique_families = sorted(set(families))
colors = sns.color_palette('tab10', len(unique_families))
family_to_color = dict(zip(unique_families, colors))

for fam in unique_families:
    mask = [f == fam for f in families]
    indices = [i for i, m in enumerate(mask) if m]
    plt.scatter(
        embeddings_2d[indices, 0],
        embeddings_2d[indices, 1],
        c=[family_to_color[fam]],
        label=fam,
        s=40,
        alpha=0.75,
        edgecolors='none'
    )

plt.xlabel('Dimensión UMAP 1', fontsize=12)
plt.ylabel('Dimensión UMAP 2', fontsize=12)
plt.title('Proyección UMAP del Espacio Latente ESM-2 (Clustering No Supervisado por Familia)', fontsize=14, fontweight='bold')
plt.legend(markerscale=1.5, fontsize=10, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(data_dir, 'umap_toxins_eda.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap de Similitud Cosetoidal Promedio Intra e Inter-Familia
families_arr = np.array(df['toxin_family'].to_list())
unique_fams = sorted(set(families_arr))

sim_matrix = np.zeros((len(unique_fams), len(unique_fams)))
for i, fam_i in enumerate(unique_fams):
    emb_i = embeddings[families_arr == fam_i]
    for j, fam_j in enumerate(unique_fams):
        emb_j = embeddings[families_arr == fam_j]
        sims = cosine_similarity(emb_i, emb_j)
        sim_matrix[i, j] = sims.mean()

plt.figure(figsize=(9, 7))
sns.heatmap(
    sim_matrix,
    xticklabels=unique_fams,
    yticklabels=unique_fams,
    annot=True,
    fmt='.3f',
    cmap='YlOrRd',
    vmin=0.5, vmax=1.0
)
plt.title('Similitud Coseno Promedio entre Familias de Toxinas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(data_dir, 'similarity_heatmap_eda.png'), dpi=150)
plt.show()

## 7. Hallazgos Preliminares y Conclusiones de la Fase 1

1. **Calidad de Datos Impecable:** El dataset curado a través de UniProt Swiss-Prot no presentó registros duplicados ni valores nulos en los atributos principales (`accession`, `sequence`, `organism`, `toxin_family`).
2. **Distribución Asimétrica Positiva en Longitud de Secuencia:** La mediana de la longitud de las secuencias de toxinas se ubica en un rango de péptidos cortos/medianos (con presencia de outliers correspondientes a enzimas o neurotoxinas complejas de mayor peso molecular).
3. **Sesgo Organísmico Representativo:** Se observa una mayor abundancia de especies pertenecientes a serpientes elápidas/vipéridas y caracoles marinos carnívoros del género *Conus*, alineado con el estado del arte de la literatura venómica.
4. **Separabilidad biológica en Espacio Latente:** La proyección UMAP y la matriz de similitud cosenoidal demuestran que las representaciones matemáticas generadas por el modelo ESM-2 agrupan naturally las toxinas por su familia funcional sin necesidad de entrenamiento supervisado.